# Credit Card Fraud Classification
<a target="_blank" href="https://colab.research.google.com/github/amiidae/showcase/blob/main/credit_card_fraud_detection.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Problem Description

In this project we are going to work on a model to predict whether a credit card transaction was fraudulent or legit.

We'll train five different classifiers from scikit-learn library:
- Logistic Regression
- Linear Support Vector Classifier
- Random Forrest Classifier
- Gradient Boosting Classifier
- AdaBoost Classifier

We also will be using cross-validation to compute three fitness metrics of these models:
- Accuracy
- F1 Score
- Area Under the ROC Curve (ROC AUC Score)


## Set Up

### Importing Libraries and Updating Settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 1000)
warnings.filterwarnings("ignore")

sns.set_theme(palette = "coolwarm", style = 'ticks', rc = {'axes.spines.left': True,
                                                           'axes.spines.bottom': True,
                                                           'axes.spines.right': False,
                                                           'axes.spines.top': False})

### About the Data

Dataset in use: [Credit Card Fraud Detection Dataset 2023](https://www.kaggle.com/datasets/nelgiriyewithana/credit-card-fraud-detection-dataset-2023) from Kaggle.

This dataset contains credit card transactions made by European cardholders in the year 2023. It comprises over 550,000 records, and the data has been anonymized, using PCA algorithm, to protect the cardholders' identities. The primary objective of this dataset is to facilitate the development of fraud detection algorithms and models to identify potentially fraudulent transactions.

Features:

- **id**: Unique identifier for each transaction.
- **V1-V28**: Anonymized features representing various transaction attributes (e.g., time, location, etc.).
- **Amount**: The transaction amount.
- **Class**: Binary label indicating whether the transaction is fraudulent (1) or not (0).

In [2]:
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1wohtgfauFBOqdIiyBaOGvt70qnWUhNnI' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1wohtgfauFBOqdIiyBaOGvt70qnWUhNnI" -O creditcard_2023.csv && rm -rf /tmp/cookies.txt

--2026-04-19 16:43:55--  https://docs.google.com/uc?export=download&confirm=&id=1wohtgfauFBOqdIiyBaOGvt70qnWUhNnI
Resolving docs.google.com (docs.google.com)... 142.250.101.101, 142.250.101.100, 142.250.101.102, ...
Connecting to docs.google.com (docs.google.com)|142.250.101.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1wohtgfauFBOqdIiyBaOGvt70qnWUhNnI&export=download [following]
--2026-04-19 16:43:55--  https://drive.usercontent.google.com/download?id=1wohtgfauFBOqdIiyBaOGvt70qnWUhNnI&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.101.132, 2607:f8b0:4023:c06::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.101.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2430 (2.4K) [text/html]
Saving to: ‘creditcard_2023.csv’

creditcard_2023.csv 100%[===================>]   2.37K  --.-KB/s   

In [3]:
df = pd.read_csv("creditcard_2023.csv", index_col = "id")

df.sample(5)

TypeError: unsupported operand type(s) for -: 'str' and 'int'

## EDA

Checking data types of the features and presence of missing values.

In [ ]:
df.info()

There are no missing values and all features are numerical - all of type float, except target feature Class, which is int.

Lets look at common statistics.

In [ ]:
df.describe()

If we look closely on `Class` column we can see, that the value of mean is 0.5.

In [ ]:
df[["Class"]].describe()

That might be the case because the labels are distributed evenly 50/50.

We can prove it by checking the number of 0 and 1 entries.

In [ ]:
num_fraudulent_transactions = df["Class"].loc[df["Class"] == 1].count()
num_legit_transactions = df["Class"].loc[df["Class"] == 0].count()

In [ ]:
print(f"""Number of fraudulent transactions: {num_fraudulent_transactions} \n
Number of legit transactions: {num_legit_transactions}""")

Dataset is perfectly symmetric and has even distribution of target variable.

### Feature Correlation

In [ ]:
correlation = df.iloc[:, :-1].corr()

In [ ]:
plt.figure(figsize = (24, 24))

mask = np.triu(np.ones_like(correlation, dtype = bool))
cmap = sns.diverging_palette(230, 20, as_cmap = True)

sns.heatmap(data = correlation, cmap = "coolwarm", mask = mask,
            linewidths = 0.5, annot = True)

plt.title("Correlation table")

plt.show()

We can see that many features are strongly correlated with each other, but we'll look only on those, which have correlation 0.7 and higher:
- `V10` with `V3`, `V4` and `V9`
- `V11` with `V4` and `V10`
- `V12` with `V3`, `V4`, `V10` and `V11`
- `V14` with `V4`, `V10`, `V11` and `V12`
- `V16` with `V12`
- `V17` with `V16`
- `V18` with `V16` and `V17`
- `V22` with `V21`

Features `V3`, `V4`, `V10`, `V11`, `V12`, `V17`, `V18` and `V21` can be removed to reduce multicollinearity.

### Feature Distributions

In [ ]:
figure = plt.figure(figsize = (20, 32))
rows, cols = 10, 3

for id, feature in enumerate(df.columns[:-1]):
    ax = figure.add_subplot(rows, cols, id + 1)
    ax.grid(alpha = 0.5, axis = "both")
    ax.set_title(f"Skewness = {df[feature].skew(axis = 0, skipna = False):.{5}}")

    sns.histplot(data = df, x = feature, hue = "Class")

    ax.set_xlabel(feature)

figure.tight_layout()
figure.show()

Some features like `V17` and `V18` have drastically different data distributions. It is important to take this into account, since this can affect performance of some algorithms.

Also, a lot of features are very skewed, especially `V7`.
We can see, that it is skewed the most.

In [ ]:
skewness = [df[feature].skew(axis = 0, skipna = False) for feature in df.columns[:-1]]
skewness = list(zip(skewness, df.columns[:-1]))

for value in sorted(skewness, reverse = True):
    print(value)

We will try to fix this with Power Transform during preprocessing.

### Outliers

In [ ]:
figure = plt.figure(figsize = (20, 32))
rows, cols = 10, 3

for id, feature in enumerate(df.columns[:-1]):
    ax = figure.add_subplot(rows, cols, id + 1)

    sns.boxplot(data = df, x = feature)

    ax.set_xlabel(feature)

figure.tight_layout()
figure.show()

Looks like almost every feature has a lot of outliers.

On this point it's hard to tell, whether it's a good idea to remove them, so, for now, lets proceed further without touching them.

## Data Preparation

Lets write a function for data preparation and create a pipeline for data preprocessing to apply before modeling.

### Cleaning Data and Removing Multicollinearity

In [ ]:
def data_preparation(dataset, columns_to_drop):

    dataset = dataset.copy().drop(columns = columns_to_drop)

    dataset = dataset.drop_duplicates()

    return dataset

In [ ]:
columns_to_drop = ["V3", "V4", "V10", "V11", "V12", "V17", "V18", "V21"]
ws = data_preparation(df, columns_to_drop)

Double checking the distribution of values in the target feature.

In [ ]:
num_fraudulent_transactions = ws["Class"].loc[ws["Class"] == 1].count()
num_legit_transactions = ws["Class"].loc[ws["Class"] == 0].count()

print(f"""Number of fraudulent transactions: {num_fraudulent_transactions} \n
Number of legit transactions: {num_legit_transactions}""")

Distribution is still perfectly even, so we can safely use simple (not stratified) K-Folds for cross-validation.

### Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer

preprocessing = Pipeline([
    ("normalize", PowerTransformer(standardize = True)),
])

## Creating Models

### Splitting Data

In [ ]:
from sklearn.model_selection import train_test_split

X = ws.iloc[:, :-1]
y = ws.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.1,
                                                    shuffle = True,
                                                    random_state = 41)

### Instantiating Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import (RandomForestClassifier,
                              GradientBoostingClassifier,
                              AdaBoostClassifier)

All models will be used in their "out-of-the-box" versions - without any parameters adjustments.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("preprocessing", preprocessing),
    ("lr_model", LogisticRegression())
])

In [ ]:
from sklearn.svm import LinearSVC

svc_model = Pipeline([
    ("preprocessing", preprocessing),
    ("svc_model", LinearSVC())
])

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc_model = Pipeline([
    ("preprocessing", preprocessing),
    ("rfc_model", RandomForestClassifier())
])

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc_model = Pipeline([
    ("preprocessing", preprocessing),
    ("gbc_model", GradientBoostingClassifier())
])

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

abc_model = Pipeline([
    ("preprocessing", preprocessing),
    ("abc_model", AdaBoostClassifier())
])

## Evaluating Models

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold

kfold = KFold(n_splits = 5, shuffle = True, random_state = 41)

metrics = ["accuracy", "f1", "roc_auc"]

lr_scores = cross_validate(lr_model, X_train, y_train, cv = kfold,
                           scoring = metrics,
                           return_train_score = True)

svc_scores = cross_validate(svc_model, X_train, y_train, cv = kfold,
                           scoring = metrics,
                           return_train_score = True)

rfc_scores = cross_validate(rfc_model, X_train, y_train, cv = kfold,
                           scoring = metrics,
                           return_train_score = True)

gbc_scores = cross_validate(gbc_model, X_train, y_train, cv = kfold,
                           scoring = metrics,
                           return_train_score = True)

abc_scores = cross_validate(abc_model, X_train, y_train, cv = kfold,
                           scoring = metrics,
                           return_train_score = True)

In [ ]:
models = {"lr_model" : lr_scores, "svc_model" : svc_scores, "rfc_model" : rfc_scores,
          "gbc_model" : gbc_scores, "abc_model" : abc_scores}


def prepare_scores(models, metrics):

  t_scores = dict(zip(models.keys(), [list() for i in range(len(models))]))
  v_scores = dict(zip(models.keys(), [list() for i in range(len(models))]))
  train_time = dict(zip(models.keys(), [list() for i in range(len(models))]))

  for name, model in models.items():
      train_scores = [np.array(model["train_" + metric]).mean() for metric in metrics]
      train_scores = dict(zip(metrics, train_scores))
      t_scores[name] = train_scores

      validation_scores = [np.array(model["test_" + metric]).mean() for metric in metrics]
      validation_scores = dict(zip(metrics, validation_scores))
      v_scores[name] = validation_scores

      time = np.array(model["fit_time"]).mean()
      train_time[name] = time

  return t_scores, v_scores, train_time

t_scores, v_scores, train_time = prepare_scores(models, metrics)

## Comparing Models

In [ ]:
models = ["LogisticRegression", "LinearSVC", "RandomForestClassifier",
          "GradientBoostingClassifier", "AdaBoostClassifier"]
data = ["Train", "Validation"]

index = pd.MultiIndex.from_product([models, data], names = ["model", "data"])

metrics_comparison = pd.DataFrame(data = [t_scores["lr_model"], v_scores["lr_model"],
                               t_scores["svc_model"], v_scores["svc_model"],
                                t_scores["rfc_model"], v_scores["rfc_model"],
                                t_scores["gbc_model"], v_scores["gbc_model"],
                                t_scores["abc_model"], v_scores["abc_model"]],
                                index = index)
display(metrics_comparison)

We can see that RandomForestClassifier shows the best results.

Now lets also check model's training time.

In [ ]:
models = ["LogisticRegression", "LinearSVC", "RandomForestClassifier",
          "GradientBoostingClassifier", "AdaBoostClassifier"]

fit_time = pd.DataFrame(data = [train_time["lr_model"]/60, train_time["svc_model"]/60,
                                train_time["rfc_model"]/60, train_time["gbc_model"]/60,
                                train_time["abc_model"]/60],
                        index = models,
                        columns = ["Fit Time (min)"]).sort_values(by = "Fit Time (min)")

display(fit_time)

Although Random Forest Classifier does the best on given data, it is pretty slow - the second slowest algorithm.

At the meantime, AdaBoost Classifier is second fastest, but on 5% less accurate.

It should be considered in the process of product creation - if we value speed over accuracy - AdaBoost Classifier is a better choice, even out of the box.

But, since "speed over accuracy" is not the case in credit card fraud detection, we will accept Random Forest Classifier as the best choice and proceed further with it.

## Fine-tuning

Lets retrain Random Forest Classifier on all features and see which ones it sees as the most important.

In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.1,
                                                    shuffle = True,
                                                    random_state = 41)

In [ ]:
rfc_model.fit(X_train, y_train)

In [ ]:
sorted(list(zip(rfc_model[1].feature_importances_,rfc_model.feature_names_in_)),
       reverse = True)

Features are rated as:
- `V14`, `V10`, `V12`, `V4` - the most important.
- `V17`, `V11`, `V16`, `V3`, `V7` - less important.
- `V9`, `V2`, `V21`, `V6`, `V8`, `V27` - the least important.
- and all other features are almost insignificant at all.


Now we can perform another round of feature selection.

In [ ]:
columns_to_drop = ["V1", "V18", "V5", "V19", "V13", "V20", "V28",
                   "V26", "V15", "V23", "V25", "V24", "V22", "Amount"]
ws = data_preparation(df, columns_to_drop)

In [ ]:
X = ws.iloc[:, :-1]
y = ws.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.1,
                                                    shuffle = True,
                                                    random_state = 41)

In [ ]:
rfc_scores = cross_validate(rfc_model, X_train, y_train, cv = kfold,
                            scoring = metrics,
                            return_train_score = True)

rfc_t_score, rfc_v_score, rfc_train_time = prepare_scores({"rfc_model" : rfc_scores}, metrics)

In [ ]:
models = ["RandomForestClassifier"]
data = ["Train", "Validation"]

index = pd.MultiIndex.from_product([models, data], names = ["model", "data"])

metrics_comparison = pd.DataFrame(data = [rfc_t_score["rfc_model"],
                               rfc_v_score["rfc_model"]],
                               index = index)
display(metrics_comparison)

Results are still impressive, considering that almost half of features was dropped.

## Analyzing Model's Performance

We can finally retrain our model on all available data and measure it's generalization error.

In [ ]:
rfc_model.fit(X_train, y_train)

In [ ]:
def eval_test(model, X_test, y_test):
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

    metrics = ["accuracy", "f1", "roc_auc"]

    test_predictions = model.predict(X_test)

    test_scores = list((accuracy_score(y_test, test_predictions),
                        f1_score(y_test, test_predictions),
                        roc_auc_score(y_test, test_predictions)))

    test_scores = dict(zip(metrics, test_scores))

    return test_scores, test_predictions

In [ ]:
rfc_test_score, rfc_test_predictions = eval_test(rfc_model, X_test, y_test)

In [ ]:
models = ["RandomForestClassifier"]
data = ["Test"]

index = pd.MultiIndex.from_product([models, data], names = ["model", "data"])

metrics_comparison = pd.DataFrame(data = [rfc_test_score] , index = index)
display(metrics_comparison)

99% accuracy and both F1 and ROC AUC Scores of almost 1.
Metrics are even slightly higher than on the Validation Sets.

Looks like the case for publishing a paper.

## Analyzing Errors

Lets check out if the outliers affected model's performance much.

Detecting misclassified data entries.

In [ ]:
mis_class_ids = y_test.loc[y_test != rfc_test_predictions].index

mf = X_test.loc[mis_class_ids]

In [ ]:
num_misclassified = mf.shape[0]
print(f"Number of misclassified data entries: {num_misclassified}")

Detecting outliers by using IQR.

In [ ]:
def get_outliers_ids(dataset):

    num_type_features = dataset.loc[:, dataset.dtypes != object].columns

    outliers_ids = dict(zip(num_type_features,
                        [list() for i in range(len(num_type_features))]))

    for feature in num_type_features:

        q1 = np.percentile(dataset[feature], 25)
        q3 = np.percentile(dataset[feature], 75)
        iqr_range = (q3 - q1) * 1.5

        lower_bound = q1 - iqr_range
        upper_bound = q3 + iqr_range

        outliers = dataset.loc[(dataset[feature] < lower_bound) |
                               (dataset[feature] > upper_bound)].index

        outliers_ids[feature].extend(outliers)

    combined_outliers_ids = set()

    for key in outliers_ids:
        combined_outliers_ids.update(outliers_ids[key])

    return outliers_ids, combined_outliers_ids

In [ ]:
outliers_ids, combined_outliers_ids = get_outliers_ids(ws)

In [ ]:
num_outliers = len(combined_outliers_ids)
print(f"Number of outliers: {num_outliers}")

Reviewing how many of misclassified data entries were among outliers.

In [ ]:
num_outliers_in_mis_class = mf.loc[mf.index.isin(combined_outliers_ids)].shape[0]
print(f"Number of outliers in misclassified data entries: {num_outliers_in_mis_class}")

Only 2 of 9 misclassified data entries were outliers, so it's not much of a problem, that they were not removed.

# Conclusions

As we were managed to prove, it's definitely possible to build a good classifier on anonymized data.

But there are some major problems in working with that kind of data.

1. Anonymized features do not give a chance to detect whether data leakage took place and prevent it.
2. Anonymized features do not give a chance to perform meaningful error analysis.
3. With anonymized data (especially preprocessed by PCA algorithm) it is impossible to know how hard or easy it is to obtain the features on which model is built on. For example, the model is build on a handful of most promising features. Than one day this features are considered too expensive to extract, or considered the source of data leakage, and therefore they are withdrawn. In that case all process of creating a model must be started over, that might not have been the case if features were known beforehand.